
# ITBI residencial no Rio de Janeiro — 2011 a 2024

**Objetivo:** unificar as bases do ITBI, selecionar as transações residenciais e obter coordenadas para a análise de mobilidade e preços imobiliários.

**Caminho seguido:** unificação → limpeza → seleção de endereços → geocodificação → verificação da precisão → exportação.

O notebook registra o tratamento da base imobiliária para posterior integração com os dados de transporte.



## 0. Preparação do ambiente

As planilhas originais do ITBI estão armazenadas em `data/raw/`. Os arquivos intermediários e os relatórios são salvos em `data/intermediate/itbi/`, enquanto as bases consolidadas utilizadas nas análises ficam em `data/processed/`. Os lotes de geocodificação e seus manifestos são preservados em `cache/geocodificacao/` para permitir sua reutilização.


In [2]:
%pip install -q pandas openpyxl geocodebr ipywidgets


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

from pathlib import Path
import hashlib
import json
import re
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

# Localizar a raiz do projeto
RAIZ = Path.cwd().resolve()

if not (RAIZ / "data").is_dir():
    RAIZ = RAIZ.parent

if not (RAIZ / "data").is_dir():
    raise FileNotFoundError(
        "Abra o notebook a partir da pasta do projeto ou de notebooks/."
    )

# Pastas de dados
PASTA_RAW = RAIZ / "data" / "raw"
PASTA_INTERMEDIARIA = RAIZ / "data" / "intermediate" / "itbi"
PASTA_PROCESSADA = RAIZ / "data" / "processed"
PASTA_CACHE = RAIZ / "cache" / "geocodificacao"

for pasta in [
    PASTA_INTERMEDIARIA,
    PASTA_PROCESSADA,
    PASTA_CACHE
]:
    pasta.mkdir(parents=True, exist_ok=True)

# Planilhas originais
ARQUIVO_2011 = PASTA_RAW / "ITBI_TRANSACOES_PERIODO_2011_2018.xlsx"
ARQUIVO_2019 = PASTA_RAW / "ITBI_TRANSACOES_PERIODO_2019_2024.xlsx"

# Compatibilidade com as células intermediárias
SAIDA = PASTA_INTERMEDIARIA

for arquivo in [ARQUIVO_2011, ARQUIVO_2019]:
    if not arquivo.is_file():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {arquivo}"
        )

print("Raiz:", RAIZ)
print("Originais:", PASTA_RAW)
print("Intermediários:", PASTA_INTERMEDIARIA)
print("Processados:", PASTA_PROCESSADA)
print("Cache:", PASTA_CACHE)


Raiz: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE
Originais: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE\data\raw
Intermediários: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE\data\intermediate\itbi
Processados: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE\data\processed
Cache: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE\cache\geocodificacao



## 1. Unificação das bases (2011–2024)

Os dois arquivos possuem a mesma estrutura de colunas e foram reunidos em uma única tabela.

Adicionamos `periodo_base`, `linha_excel` e `id_transacao` para identificar a origem de cada registro.


In [4]:
# Ler os dois Excel diretamente (sem depender do itbi.xlsx nem de CSVs de outra execução).
a = pd.read_excel(ARQUIVO_2011, dtype={"numero": "string"})
b = pd.read_excel(ARQUIVO_2019, dtype={"numero": "string"})

if set(a.columns) != set(b.columns):
    raise ValueError(
        "As colunas dos dois períodos não coincidem. "
        f"Só em 2011–2018: {sorted(set(a.columns)-set(b.columns))}; "
        f"só em 2019–2024: {sorted(set(b.columns)-set(a.columns))}"
    )
b = b[a.columns]  # Mesma ordem de colunas.

for tabela, periodo in ((a, "2011-2018"), (b, "2019-2024")):
    tabela["periodo_base"] = periodo
    tabela["linha_excel"] = np.arange(2, len(tabela) + 2)  # Cabeçalho na linha 1.
    tabela["id_transacao"] = (periodo + "_" + tabela["linha_excel"].astype(str))

itbi_unificado = pd.concat([a, b], ignore_index=True)
del a, b

assert itbi_unificado["id_transacao"].is_unique
print("Total unificado:", len(itbi_unificado))
print(itbi_unificado.groupby("periodo_base").size().to_string())

# Conferir os anos com base na coluna numérica (não em 'ano_texto' codificado pelo Excel).
itbi_unificado["ano"] = pd.to_numeric(itbi_unificado["ano"], errors="coerce")
itbi_unificado["mes"] = pd.to_numeric(itbi_unificado["mes"], errors="coerce")
anos = sorted(itbi_unificado["ano"].dropna().astype(int).unique().tolist())
print("Anos observados:", anos)
if not set(anos).issubset(set(range(2011, 2025))):
    print("ATENÇÃO: a base contém registros com ano fora de 2011–2024; verificar.")
if set(range(2011, 2025)) - set(anos):
    print("ATENÇÃO: anos ausentes:", sorted(set(range(2011, 2025)) - set(anos)))

display(itbi_unificado[["id_transacao", "ano", "mes", "periodo_base", "logradouro", "numero", "area", "valor_m2"]].head())

c:\Users\Casa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Casa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Total unificado: 532774
periodo_base
2011-2018    289110
2019-2024    243664
Anos observados: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


,id_transacao,ano,mes,periodo_base,logradouro,numero,area,valor_m2
0,2011-2018_2,2011,7,2011-2018,Avenida_x0020_Brasil,_x0032_3406,49.0,1510.204082
1,2011-2018_3,2011,7,2011-2018,Rua_x0020_Leopoldina_x0020_Borges,_x0039_5,66.0,2121.212121
2,2011-2018_4,2011,7,2011-2018,Rua_x0020_Cabo_x0020_Honório_x0020_de_x0020_Ol...,_x0037_2,92.0,869.565217
3,2011-2018_5,2011,7,2011-2018,RUA_x0020_ASSURINIS,_x0031_0,50.0,2800.000000
4,2011-2018_6,2011,7,2011-2018,Rua_x0020_Sebastião_x0020_Caldas,_x0031_10,45.0,2111.111111



## 2. Limpeza e seleção dos imóveis residenciais

Padronizamos os campos de texto e corrigimos problemas de formatação dos arquivos Excel.

Selecionamos as transações classificadas como residenciais, com tipologia de casa ou apartamento.

Os valores monetários e as áreas foram preservados.


In [5]:
def decodificar_excel(valor):
    if pd.isna(valor):
        return ""
    s = str(valor)
    s = re.sub(
        r"_x([0-9A-Fa-f]{4})_",
        lambda m: chr(int(m.group(1), 16)),
        s,
        flags=re.IGNORECASE,
    )
    return re.sub(r"\s+", " ", s).strip()

itbi = itbi_unificado.copy()
COLUNAS_TEXTO = [
    "ano_texto", "mes_texto", "cod_tipologia", "numero",
    "COMPLEMENTO", "nm_bairro_norm", "logradouro",
    "ds_imovel_tipologia", "DESCRICAO", "parte_transferida",
]
for coluna in COLUNAS_TEXTO:
    itbi[coluna] = itbi[coluna].map(decodificar_excel)

for coluna in ("logradouro", "numero", "nm_bairro_norm", "ds_imovel_tipologia", "DESCRICAO"):
    itbi[coluna] = itbi[coluna].str.upper()

# Excel eventualmente transforma o número 120 em '120.0'.
itbi["numero"] = itbi["numero"].str.replace(r"^(\d+)\.0$", r"\1", regex=True)

TIPOLOGIAS_RESIDENCIAIS = {
    "APARTAMENTO", "CASA", "PROLETARIO CASA/APTO",
    "PROLETARIO APARTAMENTO", "PROLETARIO CASA",
}

candidatos = itbi[itbi["DESCRICAO"] == "RESIDENCIAL"].copy()
itbi_residencial = candidatos[
    candidatos["ds_imovel_tipologia"].isin(TIPOLOGIAS_RESIDENCIAIS)
].copy()
excluidos_tipologia = candidatos[
    ~candidatos["ds_imovel_tipologia"].isin(TIPOLOGIAS_RESIDENCIAIS)
].copy()

assert itbi_residencial["id_transacao"].is_unique
print("Todas as transações:", len(itbi))
print("Residenciais (casas/apartamentos):", len(itbi_residencial))
print("Outras tipologias classificadas como residenciais:", len(excluidos_tipologia))
print("\nTransações residenciais por ano:")
display(itbi_residencial.groupby("ano", dropna=False).size().rename("transacoes").to_frame())

Todas as transações: 532774
Residenciais (casas/apartamentos): 441713
Outras tipologias classificadas como residenciais: 5713

Transações residenciais por ano:


,transacoes
ano,
2011,21298
2012,39520
2013,37858
2014,31578
2015,27354
2016,22961
2017,27176
2018,24574
2019,25614



## 3. Verificação dos endereços

Para a geocodificação, exigimos logradouro, número e bairro.

Os registros incompletos foram separados em `df_revisao`, sem serem eliminados da base residencial.

Nesta etapa, não aplicamos filtros sobre os valores dos imóveis.


In [6]:
AUSENTES = {
    "", "NAN", "NONE", "NULL", "N/A", "-", "IGNORADO",
    "NAO INFORMADO", "NÃO INFORMADO", "SEM INFORMACAO", "SEM INFORMAÇÃO",
}

sem_logradouro = itbi_residencial["logradouro"].isin(AUSENTES)
sem_bairro = itbi_residencial["nm_bairro_norm"].isin(AUSENTES)
sem_numero = (
    itbi_residencial["numero"].isin(AUSENTES | {"S/N", "SN", "SEM NUMERO", "SEM NÚMERO"})
    | itbi_residencial["numero"].str.fullmatch(r"0+", na=False)
)

itbi_residencial["status_limpeza"] = np.where(
    sem_logradouro | sem_bairro | sem_numero, "revisao", "valido"
)
itbi_residencial["motivo_revisao"] = (
    np.where(sem_logradouro, "sem_logradouro; ", "")
    + np.where(sem_numero, "sem_numero; ", "")
    + np.where(sem_bairro, "sem_bairro; ", "")
)
itbi_residencial["motivo_revisao"] = itbi_residencial["motivo_revisao"].str.rstrip("; ")

df_validos = itbi_residencial.loc[itbi_residencial["status_limpeza"] == "valido"].copy()
df_revisao = itbi_residencial.loc[itbi_residencial["status_limpeza"] != "valido"].copy()

assert len(df_validos) + len(df_revisao) == len(itbi_residencial)
print("Com endereço completo:", len(df_validos))
print("Para revisão de endereço:", len(df_revisao))
print(df_revisao["motivo_revisao"].value_counts().head(10))

Com endereço completo: 440495
Para revisão de endereço: 1218
motivo_revisao
sem_numero                                809
sem_logradouro                            309
sem_bairro                                 94
sem_logradouro; sem_bairro                  4
sem_logradouro; sem_numero; sem_bairro      1
sem_logradouro; sem_numero                  1
Name: count, dtype: int64



## 4. Identificação de endereços únicos

Criamos uma chave usando logradouro, número e bairro, com padronização de acentos, maiúsculas e espaços.

Endereços repetidos entre 2011 e 2024 foram agrupados para evitar geocodificações desnecessárias.

**Observação:** a deduplicação ocorre apenas nos endereços, não nas transações imobiliárias.


In [7]:
def normalizar_parte(valor):
    s = decodificar_excel(valor).upper()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip()

def chave_de_endereco(logradouro, numero, bairro):
    n = re.sub(r"^(\d+)\.0$", r"\1", normalizar_parte(numero))
    return "|".join((normalizar_parte(logradouro), n, normalizar_parte(bairro)))

df_validos["chave_endereco"] = [
    chave_de_endereco(l, n, b)
    for l, n, b in zip(df_validos["logradouro"], df_validos["numero"], df_validos["nm_bairro_norm"])
]

# A ordenação por chave torna o identificador determinístico para esta lista de endereços.
enderecos = (
    df_validos.sort_values("id_transacao")
    .drop_duplicates("chave_endereco", keep="first")
    [["chave_endereco", "logradouro", "numero", "nm_bairro_norm"]]
    .sort_values("chave_endereco")
    .reset_index(drop=True)
)
enderecos.insert(0, "id_endereco", np.arange(1, len(enderecos) + 1))

n_transacoes = df_validos.groupby("chave_endereco").size()
anos_end = df_validos.groupby("chave_endereco")["ano"].agg(["min", "max"])
enderecos["quantidade_transacoes"] = enderecos["chave_endereco"].map(n_transacoes)
enderecos["primeiro_ano"] = enderecos["chave_endereco"].map(anos_end["min"])
enderecos["ultimo_ano"] = enderecos["chave_endereco"].map(anos_end["max"])
enderecos["endereco_busca"] = (
    enderecos["logradouro"] + ", " + enderecos["numero"] + ", "
    + enderecos["nm_bairro_norm"] + ", RIO DE JANEIRO, RJ, BRASIL"
)

df_validos = df_validos.merge(
    enderecos[["chave_endereco", "id_endereco", "endereco_busca"]],
    on="chave_endereco", how="left", validate="many_to_one"
)
assert len(df_validos) + len(df_revisao) == len(itbi_residencial)
assert df_validos["id_transacao"].is_unique
assert enderecos["chave_endereco"].is_unique
assert df_validos["id_endereco"].notna().all()

print("Transações residenciais completas:", len(df_validos))
print("Endereços únicos de 2011–2024:", len(enderecos))
print("Consultas repetidas evitadas:", len(df_validos) - len(enderecos))
display(enderecos.head(10))

Transações residenciais completas: 440495
Endereços únicos de 2011–2024: 69959
Consultas repetidas evitadas: 370536


,id_endereco,chave_endereco,logradouro,numero,nm_bairro_norm,quantidade_transacoes,primeiro_ano,ultimo_ano,endereco_busca
0,1,ALAMEDA MINISTRO GAMA FILHO|43|SAO FRANCISCO X...,ALAMEDA MINISTRO GAMA FILHO,43,SAO FRANCISCO XAVIER,1,2011,2011,"ALAMEDA MINISTRO GAMA FILHO, 43, SAO FRANCISCO..."
1,2,ALM A PAL 17906|380|RECREIO DOS BANDEIRANTES,ALM A PAL 17906,380,RECREIO DOS BANDEIRANTES,4,2011,2022,"ALM A PAL 17906, 380, RECREIO DOS BANDEIRANTES..."
2,3,ALM SECUNDARIA|14|PENHA,ALM SECUNDARIA,14,PENHA,1,2017,2017,"ALM SECUNDARIA, 14, PENHA, RIO DE JANEIRO, RJ,..."
3,4,AVENIDA A PAA 10090/PAL 37215|323|ITANHANGA,AVENIDA A PAA 10090/PAL 37215,323,ITANHANGA,249,2014,2014,"AVENIDA A PAA 10090/PAL 37215, 323, ITANHANGA,..."
4,5,AVENIDA ABRAHAM MEDINA|355|BARRA DA TIJUCA,AVENIDA ABRAHAM MEDINA,355,BARRA DA TIJUCA,127,2015,2022,"AVENIDA ABRAHAM MEDINA, 355, BARRA DA TIJUCA, ..."
5,6,AVENIDA ABRAHAM MEDINA|355|CAMORIM,AVENIDA ABRAHAM MEDINA,355,CAMORIM,213,2022,2024,"AVENIDA ABRAHAM MEDINA, 355, CAMORIM, RIO DE J..."
6,7,AVENIDA ACRISIO MOTA|122|GUADALUPE,AVENIDA ACRÍSIO MOTA,122,GUADALUPE,1,2018,2018,"AVENIDA ACRÍSIO MOTA, 122, GUADALUPE, RIO DE J..."
7,8,AVENIDA ACRISIO MOTA|181|GUADALUPE,AVENIDA ACRÍSIO MOTA,181,GUADALUPE,1,2019,2019,"AVENIDA ACRÍSIO MOTA, 181, GUADALUPE, RIO DE J..."
8,9,AVENIDA ACRISIO MOTA|191|GUADALUPE,AVENIDA ACRÍSIO MOTA,191,GUADALUPE,2,2014,2015,"AVENIDA ACRÍSIO MOTA, 191, GUADALUPE, RIO DE J..."
9,10,AVENIDA ACRISIO MOTA|211|GUADALUPE,AVENIDA ACRÍSIO MOTA,211,GUADALUPE,1,2016,2016,"AVENIDA ACRÍSIO MOTA, 211, GUADALUPE, RIO DE J..."


In [8]:
# Salvar os intermediários ANTES de qualquer geocodificação.
# CSVs evitam o erro ArrowKeyError de gravação em Parquet apresentado anteriormente.

intermediarios = {
    "itbi_unificado_todas_tipologias_2011_2024.csv": itbi,
    "itbi_residencial_todos_2011_2024.csv": itbi_residencial,
    "itbi_residencial_endereco_completo_2011_2024.csv": df_validos,
    "itbi_enderecos_unicos_2011_2024.csv": enderecos,
    "itbi_residencial_revisao_2011_2024.csv": df_revisao,
    "itbi_residencial_tipologias_excluidas_2011_2024.csv": excluidos_tipologia,
}

for nome, tabela in intermediarios.items():
    tabela.to_csv(
        PASTA_INTERMEDIARIA / nome,
        index=False,
        encoding="utf-8-sig"
    )

print("Intermediários salvos em:", PASTA_INTERMEDIARIA)
print("Total de endereços únicos:", len(enderecos))


Intermediários salvos em: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE\data\intermediate\itbi
Total de endereços únicos: 69959



## 5. Reaproveitamento de coordenadas

Verificamos a possibilidade de reutilizar resultados da geocodificação anterior (2019–2024).

As correspondências são feitas pela chave completa do endereço, não pelos identificadores antigos.

Coordenadas conflitantes ou sem correspondência confiável não são reaproveitadas.


In [9]:
PASTA_ANTIGA = PASTA_CACHE / "resultados_geocodebr"
REUTILIZAR_COORDENADAS_ANTIGAS = True
COLS_GEO = ["chave_endereco", "lat", "lon", "precisao", "desvio_metros", "fonte_geocodificacao"]

reutilizados = pd.DataFrame(columns=COLS_GEO)
conflitos_antigos = []
arquivos_antigos = sorted(PASTA_ANTIGA.glob("lote_*.csv")) if (REUTILIZAR_COORDENADAS_ANTIGAS and PASTA_ANTIGA.is_dir()) else []

if arquivos_antigos:
    partes = []
    for caminho in arquivos_antigos:
        antigo = pd.read_csv(caminho, dtype={"numero": "string"}, low_memory=False)
        bairro_col = "nm_bairro_norm" if "nm_bairro_norm" in antigo.columns else (
            "localidade" if "localidade" in antigo.columns else None
        )
        necessarias = {"logradouro", "numero", "lat", "lon"}
        if bairro_col is None or not necessarias.issubset(antigo.columns):
            print("Ignorando arquivo antigo sem campos para conferir o endereço:", caminho.name)
            continue
        antigo["chave_endereco"] = [
            chave_de_endereco(l, n, b)
            for l, n, b in zip(antigo["logradouro"], antigo["numero"], antigo[bairro_col])
        ]
        for coluna in ("lat", "lon", "desvio_metros"):
            if coluna not in antigo.columns:
                antigo[coluna] = np.nan
            antigo[coluna] = pd.to_numeric(antigo[coluna], errors="coerce")
        if "precisao" not in antigo.columns:
            antigo["precisao"] = None
        antigo = antigo.loc[
            antigo["lat"].between(-90, 90)
            & antigo["lon"].between(-180, 180)
            & antigo["chave_endereco"].isin(enderecos["chave_endereco"])
        ].copy()
        partes.append(antigo[["chave_endereco", "lat", "lon", "precisao", "desvio_metros"]])

    if partes:
        antigos = pd.concat(partes, ignore_index=True)
        # Um mesmo endereço com coordenadas distintas requer revisão, não fusão automática.
        pares_distintos = antigos.drop_duplicates(["chave_endereco", "lat", "lon"])
        contagens = pares_distintos.groupby("chave_endereco").size()
        conflitos_antigos = contagens[contagens > 1].index.tolist()
        reutilizados = antigos[
            ~antigos["chave_endereco"].isin(conflitos_antigos)
        ].drop_duplicates("chave_endereco")
        reutilizados = reutilizados.copy()
        reutilizados["fonte_geocodificacao"] = "geocodebr_lotes_2019_2024"
        reutilizados.to_csv(SAIDA / "coordenadas_reutilizadas_2019_2024.csv", index=False, encoding="utf-8-sig")

print("Endereços reaproveitados de lotes antigos:", len(reutilizados))
print("Chaves antigas ambíguas, não reaproveitadas:", len(conflitos_antigos))
print("Endereços ainda sem coordenada reaproveitada:", len(enderecos) - len(reutilizados))

Endereços reaproveitados de lotes antigos: 0
Chaves antigas ambíguas, não reaproveitadas: 0
Endereços ainda sem coordenada reaproveitada: 69959



## 6. Geocodificação com geocodebr

Utilizamos o `geocodebr`, baseado no CNEFE/IBGE, para localizar os endereços ainda sem coordenadas.

O processamento foi dividido em lotes de 2.000 endereços, com resultados salvos em CSV.

Os lotes já concluídos podem ser reaproveitados caso a execução seja interrompida.


In [10]:
from geocodebr import definir_campos, geocode
from importlib.metadata import version, PackageNotFoundError
try:
    VERSAO_GEOCODEBR = version("geocodebr")
except PackageNotFoundError:
    VERSAO_GEOCODEBR = "versao_nao_identificada"

TAMANHO_LOTE = 2000
# Opcional: reduza para 100 num primeiro ensaio, mas isso cria outra pasta de lotes.

campos = definir_campos(
    logradouro="logradouro", numero="numero", localidade="localidade",
    municipio="municipio", estado="estado"
)

lista_chaves = "\n".join(enderecos["chave_endereco"].tolist())
assinatura = hashlib.sha256(lista_chaves.encode("utf-8")).hexdigest()
PASTA_LOTES = PASTA_CACHE / f"lotes_{assinatura[:12]}_n{TAMANHO_LOTE}"
PASTA_LOTES.mkdir(parents=True, exist_ok=True)

manifesto = {
    "assinatura_enderecos": assinatura,
    "quantidade_enderecos": int(len(enderecos)),
    "tamanho_lote": TAMANHO_LOTE,
    "versao_geocodebr": VERSAO_GEOCODEBR,
    "criterio_chave": "logradouro|numero|bairro sem acentos, maiusculo",
}
manifesto_path = PASTA_LOTES / "manifesto.json"
if manifesto_path.exists():
    anterior = json.loads(manifesto_path.read_text(encoding="utf-8"))
    if anterior != manifesto:
        raise RuntimeError("Manifesto incompatível. Não reutilize lotes de outra lista.")
else:
    manifesto_path.write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8")

print("Versão geocodebr:", VERSAO_GEOCODEBR)
print("Endereços únicos:", len(enderecos))
print("Coordenadas reaproveitadas:", len(reutilizados))
print("Diretório dos lotes:", PASTA_LOTES)

Versão geocodebr: 0.2.0
Endereços únicos: 69959
Coordenadas reaproveitadas: 0
Diretório dos lotes: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE\cache\geocodificacao\lotes_a919eb6cab46_n2000


In [11]:
def conferir_resultado(tabela, chaves_permitidas):
    obrigatorias = {"chave_endereco", "id_endereco", "lat", "lon", "precisao", "desvio_metros"}
    faltando = obrigatorias - set(tabela.columns)
    if faltando:
        raise ValueError(f"Colunas ausentes em resultados geocodebr: {sorted(faltando)}")
    if tabela["chave_endereco"].isna().any() or tabela["chave_endereco"].duplicated().any():
        raise ValueError("Chave de endereço nula ou duplicada no lote")
    if not set(tabela["chave_endereco"]).issubset(set(chaves_permitidas)):
        raise ValueError("Arquivo contém endereços fora do lote esperado")

novos = 0
pulados = 0
n_lotes = (len(enderecos) + TAMANHO_LOTE - 1) // TAMANHO_LOTE
chaves_reutilizadas = set(reutilizados["chave_endereco"])

for indice in range(n_lotes):
    numero_lote = indice + 1
    lote = enderecos.iloc[indice * TAMANHO_LOTE:(indice + 1) * TAMANHO_LOTE].copy()
    arquivo = PASTA_LOTES / f"lote_{numero_lote:03d}.csv"

    if arquivo.exists():
        salvo = pd.read_csv(arquivo, dtype={"numero": "string"}, low_memory=False)
        conferir_resultado(salvo, lote["chave_endereco"])
        chaves_salvas = set(salvo["chave_endereco"])
    else:
        salvo = pd.DataFrame()
        chaves_salvas = set()

    faltantes = lote.loc[
        ~lote["chave_endereco"].isin(chaves_reutilizadas | chaves_salvas)
    ].copy()
    if faltantes.empty:
        pulados += 1
        print(f"Lote {numero_lote}/{n_lotes}: nenhuma consulta pendente")
        continue

    faltantes["localidade"] = faltantes["nm_bairro_norm"]
    faltantes["municipio"] = "RIO DE JANEIRO"
    faltantes["estado"] = "RJ"
    entrada = PASTA_LOTES / f"entrada_{numero_lote:03d}.csv"
    faltantes.to_csv(entrada, index=False, encoding="utf-8-sig")
    print(f"Lote {numero_lote}/{n_lotes}: geocodificando {len(faltantes)} endereços")

    try:
        resposta = geocode(
            enderecos=str(entrada), campos_endereco=campos,
            resultado_completo=True, resolver_empates=True,
            verboso=True, cache=True
        ).to_pandas()
    finally:
        # Não acumular entradas temporárias de processamento.
        entrada.unlink(missing_ok=True)

    conferir_resultado(resposta, faltantes["chave_endereco"])
    if set(resposta["chave_endereco"]) != set(faltantes["chave_endereco"]):
        raise RuntimeError(f"Geocodebr não retornou exatamente as chaves do lote {numero_lote}")
    if len(resposta) != len(faltantes):
        raise RuntimeError(f"Quantidade de linhas divergente no lote {numero_lote}")

    resposta["fonte_geocodificacao"] = "geocodebr_cnefe"
    combinado = pd.concat([salvo, resposta], ignore_index=True) if len(salvo) else resposta
    conferir_resultado(combinado, lote["chave_endereco"])

    temporario = PASTA_LOTES / f"lote_{numero_lote:03d}.tmp.csv"
    combinado.to_csv(temporario, index=False, encoding="utf-8-sig")
    teste_salvo = pd.read_csv(temporario, dtype={"numero": "string"}, low_memory=False)
    conferir_resultado(teste_salvo, lote["chave_endereco"])
    if set(teste_salvo["chave_endereco"]) != set(combinado["chave_endereco"]):
        raise RuntimeError("Problema ao salvar o CSV temporário")
    temporario.replace(arquivo)
    novos += len(resposta)
    print(f"  CSV do lote {numero_lote} salvo ({len(combinado)} registros).")

print("Lotes sem consultas novas:", pulados)
print("Endereços geocodificados nesta execução:", novos)

Lote 1/35: nenhuma consulta pendente
Lote 2/35: nenhuma consulta pendente
Lote 3/35: nenhuma consulta pendente
Lote 4/35: nenhuma consulta pendente
Lote 5/35: nenhuma consulta pendente
Lote 6/35: nenhuma consulta pendente
Lote 7/35: nenhuma consulta pendente
Lote 8/35: nenhuma consulta pendente
Lote 9/35: nenhuma consulta pendente
Lote 10/35: nenhuma consulta pendente
Lote 11/35: nenhuma consulta pendente
Lote 12/35: nenhuma consulta pendente
Lote 13/35: nenhuma consulta pendente
Lote 14/35: nenhuma consulta pendente
Lote 15/35: nenhuma consulta pendente
Lote 16/35: nenhuma consulta pendente
Lote 17/35: nenhuma consulta pendente
Lote 18/35: nenhuma consulta pendente
Lote 19/35: nenhuma consulta pendente
Lote 20/35: nenhuma consulta pendente
Lote 21/35: nenhuma consulta pendente
Lote 22/35: nenhuma consulta pendente
Lote 23/35: nenhuma consulta pendente
Lote 24/35: nenhuma consulta pendente
Lote 25/35: nenhuma consulta pendente
Lote 26/35: nenhuma consulta pendente
Lote 27/35: nenhuma c


## 7. Avaliação da precisão geográfica

Classificamos os resultados da geocodificação conforme o nível de localização e o desvio estimado.

Para a análise espacial, adotamos como alta precisão os endereços localizados por número ou número aproximado, com desvio estimado de até 30 metros.

Também identificamos resultados de precisão intermediária, baixa precisão, sem coordenadas ou que precisam de revisão espacial.

**Limitação:** o desvio informado não garante a localização exata do imóvel.


In [12]:
# Ler TODOS os lotes gerados para esta lista de endereços.
partes = []
for indice in range(n_lotes):
    arquivo = PASTA_LOTES / f"lote_{indice + 1:03d}.csv"
    if arquivo.exists():
        parte = pd.read_csv(arquivo, dtype={"numero": "string"}, low_memory=False)
        lote = enderecos.iloc[indice * TAMANHO_LOTE:(indice + 1) * TAMANHO_LOTE]
        conferir_resultado(parte, lote["chave_endereco"])
        partes.append(parte)

novos_todos = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=COLS_GEO)
if not novos_todos.empty and "fonte_geocodificacao" not in novos_todos.columns:
    novos_todos["fonte_geocodificacao"] = "geocodebr_cnefe"

# Se o mesmo endereço existir nos dois locais, preferir o resultado do processamento novo.
consolidados = pd.concat(
    [novos_todos[COLS_GEO], reutilizados[COLS_GEO]], ignore_index=True
).drop_duplicates("chave_endereco", keep="first")

faltantes = set(enderecos["chave_endereco"]) - set(consolidados["chave_endereco"])
if faltantes:
    raise RuntimeError(
        f"Faltam resultados para {len(faltantes)} endereços. "
        "Execute novamente a célula de geocodificação antes de continuar."
    )

resultados = enderecos.merge(consolidados, on="chave_endereco", how="left", validate="one_to_one")
assert len(resultados) == len(enderecos)

for coluna in ["lat", "lon", "desvio_metros"]:
    resultados[coluna] = pd.to_numeric(resultados[coluna], errors="coerce")

coord = resultados["lat"].between(-90, 90) & resultados["lon"].between(-180, 180)
plausivel_rio = resultados["lat"].between(-23.5, -22.5) & resultados["lon"].between(-44.0, -42.5)
numero = resultados["precisao"].isin(["numero", "numero_aproximado"])
ate_30m = resultados["desvio_metros"].le(30)

resultados["qualidade_geocodificacao"] = "baixa_precisao"
resultados.loc[~coord, "qualidade_geocodificacao"] = "sem_coordenada"
resultados.loc[coord & numero, "qualidade_geocodificacao"] = "precisao_intermediaria"
resultados.loc[coord & numero & ate_30m, "qualidade_geocodificacao"] = "alta_precisao"
resultados.loc[coord & ~plausivel_rio, "qualidade_geocodificacao"] = "revisao_espacial"

resultados["lat_analise"] = resultados["lat"].where(
    resultados["qualidade_geocodificacao"] == "alta_precisao"
)
resultados["lon_analise"] = resultados["lon"].where(
    resultados["qualidade_geocodificacao"] == "alta_precisao"
)


resultados.to_csv(
    PASTA_INTERMEDIARIA / "itbi_enderecos_geocodificados_2011_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Endereços totais:", len(resultados))
print("\nQualidade da geocodificação por endereço:")
print(resultados["qualidade_geocodificacao"].value_counts(dropna=False).to_string())
display(resultados[["endereco_busca", "lat", "lon", "precisao", "desvio_metros", "qualidade_geocodificacao"]].head())

Endereços totais: 69959

Qualidade da geocodificação por endereço:
qualidade_geocodificacao
alta_precisao             62944
baixa_precisao             3852
precisao_intermediaria     3163


,endereco_busca,lat,lon,precisao,desvio_metros,qualidade_geocodificacao
0,"ALAMEDA MINISTRO GAMA FILHO, 43, SAO FRANCISCO...",-22.901083,-43.242611,numero,6,alta_precisao
1,"ALM A PAL 17906, 380, RECREIO DOS BANDEIRANTES...",-23.017258,-43.470551,localidade,6179,baixa_precisao
2,"ALM SECUNDARIA, 14, PENHA, RIO DE JANEIRO, RJ,...",-22.841560,-43.279930,localidade,1694,baixa_precisao
3,"AVENIDA A PAA 10090/PAL 37215, 323, ITANHANGA,...",-22.987846,-43.313755,localidade,2197,baixa_precisao
4,"AVENIDA ABRAHAM MEDINA, 355, BARRA DA TIJUCA, ...",-22.983549,-43.415092,numero,91,precisao_intermediaria



## 8. Associação das coordenadas às transações

As coordenadas obtidas foram associadas às transações pela chave de endereço.

Os registros sem geocodificação foram mantidos na base completa, com indicação da situação.

Criamos `id_grupo_geografico` para identificar registros com coordenadas iguais, sem assumir que representam o mesmo imóvel.


In [13]:
colunas_geo = [
    "chave_endereco", "lat", "lon", "precisao", "desvio_metros",
    "fonte_geocodificacao", "qualidade_geocodificacao", "lat_analise", "lon_analise",
]
completo = df_validos.merge(
    resultados[colunas_geo], on="chave_endereco", how="left", validate="many_to_one"
)
revisao = df_revisao.copy()
for coluna in colunas_geo:
    if coluna not in revisao.columns:
        revisao[coluna] = np.nan
revisao["qualidade_geocodificacao"] = "revisao_endereco"

base_final = pd.concat([completo, revisao], ignore_index=True)
base_final = base_final.sort_values(["periodo_base", "linha_excel"]).reset_index(drop=True)
base_final["status_geocodificacao"] = np.where(
    base_final["qualidade_geocodificacao"] == "revisao_endereco", "revisao_endereco",
    np.where(base_final["lat"].notna() & base_final["lon"].notna(), "localizado", "sem_coordenada")
)

pontos = (
    base_final[["lat", "lon"]]
    .dropna().drop_duplicates().sort_values(["lat", "lon"]).reset_index(drop=True)
)
pontos["id_grupo_geografico"] = [f"GEO{i:07d}" for i in range(1, len(pontos) + 1)]
base_final = base_final.merge(pontos, on=["lat", "lon"], how="left", validate="many_to_one")

assert len(base_final) == len(itbi_residencial)
assert base_final["id_transacao"].is_unique
assert set(base_final["id_transacao"]) == set(itbi_residencial["id_transacao"])

# Comparar colunas econômicas pela chave individual; falhar em caso de alteração inadvertida.
colunas_economicas = ["area", "valor_transacoes", "valor_m2", "ano", "mes"]
esperados = itbi_residencial.set_index("id_transacao")[colunas_economicas].sort_index()
obtidos = base_final.set_index("id_transacao")[colunas_economicas].sort_index()
pd.testing.assert_frame_equal(esperados, obtidos, check_dtype=False)



### 8.1. Recalcular o valor por metro quadrado e sinalizar possíveis erros

O valor por metro quadrado foi recalculado a partir do valor da transação dividido pela área do imóvel. Foram removidos da base analítica os registros sem preço por metro quadrado válido. Os valores extremos foram identificados separadamente para cada ano, utilizando limites de três intervalos interquartis sobre o logaritmo decimal do preço por metro quadrado.

Para preservar a rastreabilidade, foram exportadas versões com e sem os valores extremos. As análises de preços utilizarão a versão sem extremos, enquanto a versão completa será mantida para conferência. O procedimento reduz a influência de valores excepcionalmente discrepantes, mas não permite concluir que todos esses registros sejam erros de origem.

In [14]:
# Reconstruir valor_m2 exclusivamente a partir de valor_transacoes e area.
valor_original_m2 = pd.to_numeric(base_final["valor_m2"], errors="coerce")
valor_transacoes = pd.to_numeric(base_final["valor_transacoes"], errors="coerce")
area = pd.to_numeric(base_final["area"], errors="coerce")

# Impedir divisões por zero e resultados derivados de valores inválidos.
area_valida = area.notna() & np.isfinite(area) & area.gt(0)
valor_valido = valor_transacoes.notna() & np.isfinite(valor_transacoes) & valor_transacoes.gt(0)

base_final["valor_m2"] = (valor_transacoes / area).where(area_valida & valor_valido)
base_final["valor_m2"] = base_final["valor_m2"].where(np.isfinite(base_final["valor_m2"]))

base_final["status_valor_m2"] = np.select(
    [~area_valida, ~valor_valido],
    ["area_invalida_ou_ausente", "valor_transacao_invalido_ou_ausente"],
    default="calculado",
)

# Critério exploratório de valor atípico por ano: 3 x IQR de log10(valor_m2).
# Sinaliza valores suspeitos; NÃO altera valor_m2 nem descarta imóveis.
log_valor_m2 = np.log10(base_final["valor_m2"])
por_ano = log_valor_m2.groupby(base_final["ano"])
q1 = por_ano.transform(lambda s: s.quantile(0.25))
q3 = por_ano.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
base_final["valor_m2_suspeito"] = (
    (log_valor_m2 < q1 - 3 * iqr) |
    (log_valor_m2 > q3 + 3 * iqr)
).fillna(False)

# Conferência: os identificadores e a quantidade de transações não mudam.
assert len(base_final) == len(itbi_residencial)
assert base_final["id_transacao"].is_unique

# Alterações numéricas em relação à coluna antiga (desconsiderando NaN iguais).
alteracoes = ~np.isclose(
    valor_original_m2.to_numpy(dtype=float),
    base_final["valor_m2"].to_numpy(dtype=float),
    rtol=1e-10, atol=1e-8, equal_nan=True,
)
print("Transações preservadas:", len(base_final))
print("Valores recalculados:", base_final["valor_m2"].notna().sum())
print("Valores indisponíveis (sem preço ou área válidos):", base_final["valor_m2"].isna().sum())
print("Diferenças numéricas em relação aos valores anteriores:", int(alteracoes.sum()))
print("Possíveis valores atípicos para revisão:", int(base_final["valor_m2_suspeito"].sum()))
print("\nSituação do valor por m²:")
print(base_final["status_valor_m2"].value_counts(dropna=False).to_string())

# Amostra dos maiores valores para revisão da fonte.
display(base_final.nlargest(10, "valor_m2")[[
    "id_transacao", "ano", "logradouro", "numero", "area",
    "valor_transacoes", "valor_m2", "valor_m2_suspeito"
]])


Transações preservadas: 441713
Valores recalculados: 441669
Valores indisponíveis (sem preço ou área válidos): 44
Diferenças numéricas em relação aos valores anteriores: 0
Possíveis valores atípicos para revisão: 1514

Situação do valor por m²:
status_valor_m2
calculado                   441669
area_invalida_ou_ausente        44


,id_transacao,ano,logradouro,numero,area,valor_transacoes,valor_m2,valor_m2_suspeito
75001,2011-2018_92418,2013,RUA PRUDENTE DE MORAIS,699,118.0,5.670000e+09,4.805085e+07,True
94940,2011-2018_117241,2013,RUA BAMBINA,30,63.0,1.626480e+09,2.581714e+07,True
246154,2019-2024_16170,2019,PRAÇA SANTOS DUMONT,40,80.0,1.670000e+09,2.087500e+07,True
109763,2011-2018_135858,2014,RUA PROFESSOR HENRIQUE COSTA,880,57.0,9.023933e+08,1.583146e+07,True
99046,2011-2018_122465,2014,RUA ORICA,1191,69.0,4.000000e+08,5.797101e+06,True
101177,2011-2018_125063,2014,RUA LUIS CARLOS CONCEIÇÃO,28,47.0,1.200000e+08,2.553191e+06,True
82871,2011-2018_102200,2013,RUA AXIXA,0,63.0,1.480000e+08,2.349206e+06,True
101508,2011-2018_125541,2014,RUA DOUTOR NUNES,879,43.0,5.600000e+07,1.302326e+06,True
393949,2019-2024_189117,2023,RUA PAULO BARRETO,50,1.0,1.149600e+06,1.149600e+06,True
116006,2011-2018_143771,2014,AVENIDA PAULO TAPAJÓS,131,1.0,1.050157e+06,1.050157e+06,True


In [15]:

# Registrar o tamanho da base antes da exclusão
total_antes = len(base_final)

# Manter somente transações com preço por m² calculável
base_final = base_final[
    base_final["valor_m2"].notna()
    & np.isfinite(base_final["valor_m2"])
    & (base_final["valor_m2"] > 0)
].copy()

print("Registros antes:", total_antes)
print("Registros depois:", len(base_final))
print("Registros excluídos:", total_antes - len(base_final))


Registros antes: 441713
Registros depois: 441669
Registros excluídos: 44


In [18]:

# Base residencial com preços válidos
# Inclui os valores extremos sinalizados.
base_com_extremos = base_final.copy()

# Base residencial sem os valores extremos sinalizados.
base_sem_extremos = base_final.loc[
    ~base_final["valor_m2_suspeito"]
].copy()

# Base espacial de alta precisão, com extremos.
alta_com_extremos = base_com_extremos.loc[
    base_com_extremos["qualidade_geocodificacao"] == "alta_precisao"
].copy()

# Base espacial de alta precisão, sem extremos.
alta_sem_extremos = base_sem_extremos.loc[
    base_sem_extremos["qualidade_geocodificacao"] == "alta_precisao"
].copy()

# 1. Completa, com extremos
base_com_extremos.to_csv(
    PASTA_PROCESSADA /
    "itbi_residencial_2011_2024_geocodificado_completo.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Completa, sem extremos
base_sem_extremos.to_csv(
    PASTA_PROCESSADA /
    "itbi_residencial_2011_2024_sem_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Alta precisão, com extremos
alta_com_extremos.to_csv(
    PASTA_PROCESSADA /
    "itbi_residencial_2011_2024_alta_precisao.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Alta precisão, sem extremos
alta_sem_extremos.to_csv(
    PASTA_PROCESSADA /
    "itbi_residencial_2011_2024_alta_precisao_sem_extremos.csv",
    index=False,
    encoding="utf-8-sig"
)

# Manter o nome 'alta' para compatibilidade com o notebook.
alta = alta_com_extremos

print("Base completa com extremos:", len(base_com_extremos))
print("Base completa sem extremos:", len(base_sem_extremos))
print("Alta precisão com extremos:", len(alta_com_extremos))
print("Alta precisão sem extremos:", len(alta_sem_extremos))
print("Extremos excluídos da análise:", len(base_com_extremos) - len(base_sem_extremos))


Base completa com extremos: 441669
Base completa sem extremos: 440155
Alta precisão com extremos: 320104
Alta precisão sem extremos: 318709
Extremos excluídos da análise: 1514


In [21]:

extremos = base_final.loc[
    base_final["valor_m2_suspeito"]
].copy()

extremos.to_csv(
    PASTA_INTERMEDIARIA /
    "itbi_valores_extremos_2011_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Transações extremas:", len(extremos))


Transações extremas: 1514



## 9. Verificações de cobertura

Geramos resumos da geocodificação por ano e bairro, incluindo as quantidades de transações localizadas e classificadas como alta precisão.

Essas informações permitem verificar se determinados períodos ou regiões ficaram menos representados na base geocodificada.


In [25]:
resumo_ano = (
    base_final.groupby("ano", dropna=False)
    .agg(
        transacoes=("id_transacao", "size"),
        enderecos_completos=("status_limpeza", lambda s: int(s.eq("valido").sum())),
        transacoes_localizadas=("lat", lambda s: int(s.notna().sum())),
        transacoes_alta_precisao=("qualidade_geocodificacao", lambda s: int(s.eq("alta_precisao").sum())),
    ).reset_index()
)
resumo_ano["pct_alta_precisao"] = (
    100 * resumo_ano["transacoes_alta_precisao"] / resumo_ano["transacoes"]
).round(2)
resumo_ano.to_csv(PASTA_INTERMEDIARIA / "relatorio_cobertura_por_ano_2011_2024.csv", index=False, encoding="utf-8-sig")

resumo_bairro = (
    base_final.groupby("nm_bairro_norm", dropna=False)
    .agg(
        transacoes=("id_transacao", "size"),
        transacoes_localizadas=("lat", lambda s: int(s.notna().sum())),
        transacoes_alta_precisao=("qualidade_geocodificacao", lambda s: int(s.eq("alta_precisao").sum())),
    ).reset_index().sort_values("transacoes", ascending=False)
)
resumo_bairro["pct_alta_precisao"] = (
    100 * resumo_bairro["transacoes_alta_precisao"] / resumo_bairro["transacoes"]
).round(2)
resumo_bairro.to_csv(PASTA_INTERMEDIARIA / "relatorio_cobertura_por_bairro_2011_2024.csv", index=False, encoding="utf-8-sig")

relatorio = {
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
    "periodo_pesquisa": "2011-2024",
    "fontes_xlsx": [ARQUIVO_2011.name, ARQUIVO_2019.name],
    "versao_geocodebr": VERSAO_GEOCODEBR,
    "versao_pandas": pd.__version__,
    "registros_todas_tipologias": int(len(itbi)),
    "transacoes_residenciais": int(len(itbi_residencial)),
    "transacoes_endereco_incompleto": int(len(df_revisao)),
    "transacoes_endereco_completo": int(len(df_validos)),
    "enderecos_unicos": int(len(enderecos)),
    "enderecos_reaproveitados_do_cache_antigo": int(len(reutilizados)),
    "enderecos_reutilizacao_ambigua": int(len(conflitos_antigos)),
    "enderecos_com_coordenadas": int(resultados["lat"].notna().sum()),
    "enderecos_alta_precisao": int(resultados["qualidade_geocodificacao"].eq("alta_precisao").sum()),
    "transacoes_com_coordenadas": int(base_final["lat"].notna().sum()),
    "transacoes_alta_precisao": int(base_final["qualidade_geocodificacao"].eq("alta_precisao").sum()),
    "criterio_alta_precisao": "numero ou numero_aproximado + desvio_metros <= 30 m + coordenada plausivel",
    "assinatura_lista_enderecos": assinatura,
    "transacoes_residenciais_originais": int(len(itbi_residencial)),
    "transacoes_com_preco_valido": int(len(base_com_extremos)),
    "transacoes_sem_preco_valido": int(
        len(itbi_residencial) - len(base_com_extremos)
    ),
    "transacoes_com_preco_extremo": int(len(extremos)),
    "transacoes_sem_preco_extremo": int(len(base_sem_extremos)),
    "transacoes_alta_precisao_sem_extremos": int(len(alta_sem_extremos)),
    "criterio_preco_extremo": "3 x IQR de log10(valor_m2), calculado por ano",

}
(PASTA_INTERMEDIARIA / "relatorio_metodologico_2011_2024.json").write_text(
    json.dumps(relatorio, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Verificações de integridade concluídas.")
print("\nResumo por ano:")
display(resumo_ano)
print("\nArquivos disponíveis em:", SAIDA.resolve())
for caminho in sorted(PASTA_INTERMEDIARIA.glob("*.csv")):
    print(" -", caminho.name)

Verificações de integridade concluídas.

Resumo por ano:


,ano,transacoes,enderecos_completos,transacoes_localizadas,transacoes_alta_precisao,pct_alta_precisao
0,2011,21281,21208,21208,15657,73.57
1,2012,39516,39386,39386,28767,72.80
2,2013,37852,37742,37742,27047,71.45
3,2014,31576,31499,31499,22287,70.58
4,2015,27353,27286,27286,19426,71.02
5,2016,22960,22921,22921,16664,72.58
6,2017,27176,27139,27139,18511,68.12
7,2018,24573,24534,24534,16993,69.15
8,2019,25612,25564,25564,18415,71.90
9,2020,28407,28356,28356,20755,73.06



Arquivos disponíveis em: C:\Users\Casa\OneDrive\Documentos\Escola\FGV CD\aedv\TRABALHO_MOBILIDADE\data\intermediate\itbi
 - itbi_enderecos_geocodificados_2011_2024.csv
 - itbi_enderecos_unicos_2011_2024.csv
 - itbi_residencial_endereco_completo_2011_2024.csv
 - itbi_residencial_revisao_2011_2024.csv
 - itbi_residencial_tipologias_excluidas_2011_2024.csv
 - itbi_residencial_todos_2011_2024.csv
 - itbi_unificado_todas_tipologias_2011_2024.csv
 - itbi_valores_extremos_2011_2024.csv
 - relatorio_cobertura_por_ano_2011_2024.csv
 - relatorio_cobertura_por_bairro_2011_2024.csv


## 10. Arquivos gerados e limitações

O notebook produziu quatro bases residenciais com preços válidos, armazenadas em `data/processed/`:

- `itbi_residencial_2011_2024_geocodificado_completo.csv`: transações residenciais com preço válido, incluindo valores extremos.
- `itbi_residencial_2011_2024_sem_extremos.csv`: transações residenciais com preço válido, excluindo os valores extremos identificados.
- `itbi_residencial_2011_2024_alta_precisao.csv`: transações com geocodificação de alta precisão, incluindo valores extremos.
- `itbi_residencial_2011_2024_alta_precisao_sem_extremos.csv`: transações com geocodificação de alta precisão e sem valores extremos, destinada às análises principais de preços e mobilidade.

Também foram preservados os resultados intermediários e relatórios em `data/intermediate/itbi/`, incluindo `itbi_valores_extremos_2011_2024.csv`, que registra os casos sinalizados.

### Critérios e limitações

Os registros sem valor por metro quadrado válido foram removidos das bases analíticas. Os valores extremos foram identificados separadamente para cada ano por meio de três intervalos interquartis aplicados ao logaritmo decimal do valor por metro quadrado.

A identificação estatística de valores extremos não comprova que os registros sejam incorretos. Por isso, foram mantidas versões com e sem esses casos.

A análise espacial considera somente transações com geocodificação de alta precisão, conforme os critérios definidos neste notebook. Essa seleção reduz a amostra e pode afetar sua representatividade entre diferentes áreas do município.

As comparações de preços ao longo do tempo também devem considerar que os valores monetários são nominais e não foram corrigidos pela inflação.

### Continuidade da análise

A integração espacial com Metrô, BRT e VLT é realizada no `03_integracao_itbi.ipynb`. As bases finais e os resultados dessa integração serão utilizados no notebook do artigo.

### Fontes

- ITBI — Prefeitura do Rio de Janeiro, 2011–2024.
- [geocodebr — Ipea](https://ipea.github.io/geocodebr/)
- [CNEFE — IBGE](https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html)